# OpenAI Gymを使ってみる

In [15]:
# import gym
import gymnasium as gym

env = gym.make("CartPole-v1")

state = env.reset()
print(state)    # 初期状態（array[カートの位置, カートの速度, 棒の角度, 棒の角速度]

action_space = env.action_space
print(action_space) # 行動の次元数（種類）

(array([-0.03205607, -0.02839814, -0.0103127 ,  0.0305864 ], dtype=float32), {})
Discrete(2)


In [16]:
# 実際に行動を起こし、時間を一つ進めてみる
# ↓教科書に書かれていたもの↓

# action = 0  # or 1
# next_state, reward, done, info = env.step(action)
# print(next_state)

# ↓最近のGym/Gymnasiumでの実装↓
action = 0
next_state, reward, terminated, truncated, info = env.step(action)  # info はデバッグ用
done = terminated or truncated  # 教科書の書き方に合わせる
print(next_state)

[-0.03262403 -0.2233707  -0.00970098  0.3199978 ]


# ランダムなエージェント
ランダムなエージェント（ランダムに行動するエージェント）を想定し、1回分のエピソードを動かす。詳しくは`gym_play.py`を参照せよ。

# 経験再生の実装

In [ ]:
import gymnasium as gym
from replay_buffer import *

env = gym.make("CartPole-v1", render_mode="rgb_array")
replay_buffer = ReplayBuffer(buffer_size=10000, batch_size=32)

# ここからバッファを作っていく
for episode in range(10):
    # state = env.reset()
    state, info = env.reset()   # gymnasium では、env.reset()メソッドによって、リセットされた状態（ndarray）と info のタプルが返されるようになった。そのため、state には純粋な「状態」の情報のみが入るようにしてあげる。
    done = False

    while not done:
        action = 0
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        replay_buffer.add(state, action, reward, next_state, done)
        state = next_state

# 作ったバッファからミニバッチを取り出していく
state, action, reward, next_state, done = replay_buffer.get_batch()
print(state.shape)
print(action.shape)
print(reward.shape)
print(next_state.shape)
print(done.shape)

(32, 4)
(32,)
(32,)
(32, 4)
(32,)


In [20]:
# デバッグ
env = gym.make("CartPole-v1", render_mode="rgb_array")
replay_buffer = ReplayBuffer(buffer_size=10000, batch_size=32)
state = env.reset()
print(state)

action = 0
next_state, reward, terminated, truncated, info = env.step(action)
state = next_state
print(state)

(array([-0.03718367,  0.02064935, -0.0252348 , -0.01908183], dtype=float32), {})
[-0.03677069 -0.17410178 -0.02561644  0.26553363]


In [ ]:
# AIインストラクターの指示
import gymnasium as gym

env = gym.make("CartPole-v1")

# 1. env.reset()の返り値を確認
# state_reset = env.reset()
state_reset, info = env.reset() # gymnasiumの新しい仕様に合わせた。
print("【env.reset() の戻り値】")
print("型:", type(state_reset))
print("中身:", state_reset)
print("-" * 30)

# 2. env.step()の返り値（next_state）を確認
next_state, reward, terminated, truncated, info = env.step(0)
print("【env.step() の next_state】")
print("型:", type(next_state))
print("中身:", next_state)

【env.reset() の戻り値】
型: <class 'tuple'>
中身: (array([ 0.01088516,  0.04131267,  0.00967265, -0.01262752], dtype=float32), {})
------------------------------
【env.step() の next_state】
型: <class 'numpy.ndarray'>
中身: [ 0.01171142 -0.15394665  0.0094201   0.2830915 ]
